In [ ]:
"""
Validação Cruzada para NER com LLMs
=====================================
Modelos:
  - Qwen3.5-9B      : unsloth/Qwen3.5-9B         → LoRA bf16
  - Gemma 4 26B A4B : unsloth/gemma-4-26B-A4B-it  → QLoRA 4-bit via FastModel (MoE)

Hiperparâmetros definidos:
  - Épocas         : 3
  - Learning Rate  : 1e-4
  - Batch Size     : 1 por GPU (gradient_accumulation_steps=4 → batch efetivo=4)
  - Quantização    : NF4 + double quantization (load_in_4bit=True onde aplicável)
  - LoRA r         : 16
  - LoRA alpha     : 64   (escala efetiva = alpha/r = 4.0)
  - LoRA dropout   : 0.05
  - target_modules : apenas atenção — q_proj, k_proj, v_proj, o_proj

Métricas de avaliação — idênticas ao BERTimbau (compute_metrics):
  Todas calculadas via seqeval sobre sequências IOB2 completas:
  - F1 / Precision / Recall  micro  (padrão seqeval)
  - F1 / Precision / Recall  macro
  - F1 / Precision / Recall  por entidade (LOCAL, ORGANIZACAO, PESSOA, TEMPO)
  - Accuracy                 via seqeval.accuracy_score (mesma função do BERT)

Diferenças de API Unsloth por modelo:
  • Qwen3.5  → FastLanguageModel  (modelos densos)
  • Gemma 4  → FastModel          (API unificada para MoE/multimodal)

Thinking mode:
  • Qwen3.5 : /no_think no prompt + remoção de <think>...</think> no parse
  • Gemma 4 : sem token <|think|> no system prompt → thinking desabilitado por padrão;
              blocos residuais removidos via regex no parse

Fold 7 reservado para hiperparâmetros do BERT — nunca usado aqui.

Correções aplicadas:
  1. dataset_text_field corrigido; datasets de treino/teste separados por responsabilidade.
  2. entity-level removido — seqeval cobre P/R/F1 por rótulo + macro/micro avg.
  3. Acurácia via seqeval.accuracy_score — idêntica ao compute_metrics do BERT.
  4. Checkpoint robusto: JSON salvo antes de avançar o contador.
  5. LoRA aplicado apenas aos módulos de atenção conforme especificado.
  6. BitsAndBytesConfig explícito: NF4 + double quantization.
  7. Acesso seguro a campos string do HuggingFace Dataset durante inferência.
  8. entities_to_iob2: spans ordenados por comprimento (maior tem prioridade);
     colisões detectadas e reportadas em vez de sobrescrita silenciosa.
  9. warmup_steps dinâmico: evita warmup excessivo em folds pequenos.
 10. load_in_4bit removido do load_kwargs do Gemma — BitsAndBytesConfig já o define;
     passá-lo junto causava conflito com quantization_config.
 11. Dispositivo de inferência via next(model.parameters()).device — seguro com PEFT.
 12. load_data com FileNotFoundError explícito identificando qual fold falhou."""

import unsloth  # deve ser o primeiro import
import os
import gc
import re
import json
import torch
from tqdm import tqdm
from datasets import Dataset
from seqeval.metrics import (
    classification_report as seq_classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,   # mesma função usada pelo BERT
)
from transformers import BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
import numpy as np
import unicodedata          # ← adiciona aqui junto com os outros imports

# ---- função de normalização ----
def normalize(s: str) -> str:       # ← adiciona aqui, logo após os imports
    s = s.strip().lower()
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"^[^\w]+|[^\w]+$", "", s)
    return s


class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)


# ============================================================
# CONFIGURAÇÕES — altere apenas MODEL_CHOICE
# ============================================================

# "qwen"  → Qwen3.5-9B       (LoRA bf16,    ~18 GB treino)
# "gemma" → Gemma 4 26B A4B  (QLoRA 4-bit,  ~22 GB treino)
MODEL_CHOICE = "qwen"

NF4_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

MODEL_REGISTRY = {
    "qwen": {
        "hf_id":               "unsloth/Qwen3.5-9B",
        "load_in_4bit":        True,
        "quantization_config": NF4_CONFIG,
        "dtype":               None,
        "api":                 "fast_language",
    },
    "gemma": {
        "hf_id":             "unsloth/gemma-4-26B-A4B-it",
        "load_in_4bit":      True,
        "quantization_config": NF4_CONFIG,
        "dtype":             None,
        "api":               "fast_model",
    },
}

CFG        = MODEL_REGISTRY[MODEL_CHOICE]
MODEL_NAME = CFG["hf_id"]

MAX_LEN    = 1024
MAX_NEW    = 1024
NUM_EPOCHS = 3
LR         = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4

LORA_R       = 16
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

BASE_DIR = "../Partitions/Datasets/cachacaNER/"
LABELS   = [
    "NOME_BEBIDA",
    "GRADUACAO_ALCOOLICA",
    "EQUIPAMENTO_DESTILACAO", 
    "TEMPO_ARMAZENAMENTO", 
    "RECIPIENTE_ARMAZENAMENTO", 
    "TIPO_MADEIRA", 
    "CARACTERISTICA_SENSORIAL_COR", 
    "CARACTERISTICA_SENSORIAL_AROMA", 
    "CARACTERISTICA_SENSORIAL_SABOR", 
    "CARACTERISTICA_SENSORIAL_CONSISTÊNCIA", 
    "NOME_PESSOA", 
    "NOME_LOCAL", 
    "NOME_ORGANIZACAO", 
    "TEMPO", 
    "PRECO", 
    "VOLUME", 
    "CLASSIFICACAO_BEBIDA"
]


HPARAM_FOLD = 8
ALL_FOLDS   = [i for i in list(range(1, 11)) if i != HPARAM_FOLD]

CHECKPOINT_FILE = "ultimo_fold.txt"

os.makedirs("modelos_por_fold", exist_ok=True)
open("relatorio_folds.txt", "a", encoding="utf-8").close()


# ============================================================
# PROMPT
# ============================================================

def build_instruction() -> str:
    base = (
        "Extraia entidades nomeadas do texto abaixo e retorne "
        "SOMENTE um array JSON com objetos contendo os campos "
        "'text' e 'label'. Se não houver entidades, retorne [].\n"
        f"Rótulos válidos: {', '.join(LABELS)}."
    )
    return ("/no_think\n" + base) if MODEL_CHOICE == "qwen" else base


INSTRUCTION = build_instruction()


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def load_data(path: str, col_sep: str = "\t",
              token_col: int = 0, tag_col: int = -1) -> list:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[Erro] Arquivo nao encontrado: '{path}'\n"
            f"Verifique se BASE_DIR esta correto: '{BASE_DIR}'"
        )
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return _load_conll(path, col_sep=col_sep, token_col=token_col, tag_col=tag_col)


def _load_conll(path: str, col_sep: str = "\t",
                token_col: int = 0, tag_col: int = -1) -> list:
    samples  = []
    tokens   = []
    bio_tags = []

    def close_span(span_tokens, span_label, entities, span_idxs):
        if span_tokens and span_label and span_label in LABELS:
            entities.append({
                "text":          " ".join(span_tokens),
                "label":         span_label,
                "token_indices": list(span_idxs),
            })

    def flush_sentence(tokens, bio_tags):
        if not tokens:
            return
        text      = " ".join(tokens)
        entities  = []
        span_toks = []
        span_idxs = []
        span_lbl  = None

        for idx, (token, tag) in enumerate(zip(tokens, bio_tags)):
            if tag.startswith("B-"):
                close_span(span_toks, span_lbl, entities, span_idxs)
                span_toks = [token]
                span_idxs = [idx]
                span_lbl  = tag[2:]
            elif tag.startswith("I-"):
                label = tag[2:]
                if span_lbl == label:
                    span_toks.append(token)
                    span_idxs.append(idx)
                else:
                    print(f"[CoNLL] I- sem B- anterior ('{tag}' apos '{span_lbl}') em: \"{text[:80]}\"")
                    close_span(span_toks, span_lbl, entities, span_idxs)
                    span_toks = [token]
                    span_idxs = [idx]
                    span_lbl  = label
            else:
                close_span(span_toks, span_lbl, entities, span_idxs)
                span_toks = []
                span_idxs = []
                span_lbl  = None

        close_span(span_toks, span_lbl, entities, span_idxs)
        samples.append({"text": text, "tokens": list(tokens), "entities": entities})

    SENTENCE_ENDINGS = {".", "?", "!", "...", "…"}

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")

            if line.strip() == "":
                flush_sentence(tokens, bio_tags)
                tokens   = []
                bio_tags = []
                continue

            parts = line.split(col_sep)
            if len(parts) < 2:
                print(f"[CoNLL] Linha malformada ignorada: {line!r}")
                continue

            try:
                token = parts[token_col]
                tag   = parts[tag_col].strip()
            except IndexError:
                print(f"[CoNLL] Coluna ausente (token_col={token_col}, tag_col={tag_col}): {line!r}")
                continue

            tokens.append(token)
            bio_tags.append(tag)

            if token in SENTENCE_ENDINGS and tag == "O":
                prev_token = tokens[-2] if len(tokens) >= 2 else ""
                is_abbreviation = (
                    len(prev_token) <= 4
                    and prev_token[0].isupper()
                    and prev_token.isalpha()
                ) if prev_token else False
                if not is_abbreviation:
                    flush_sentence(tokens, bio_tags)
                    tokens   = []
                    bio_tags = []

    flush_sentence(tokens, bio_tags)

    print(f"\n[CoNLL] {path}: {len(samples)} sentenças carregadas.")
    print("[CoNLL] Amostra das primeiras 3 sentenças lidas:")
    for i, s in enumerate(samples[:3]):
        print(f"  Sentença {i+1}:")
        print(f"    text     : {s['text'][:80]!r}{'...' if len(s['text']) > 80 else ''}")
        print(f"    entities : {s['entities']}")
    sem_entidade = sum(1 for s in samples if not s["entities"])
    print(f"[CoNLL] Sentenças sem entidade: {sem_entidade}/{len(samples)} ({100*sem_entidade/len(samples):.1f}%)\n")

    return samples


def prepare_train_dataset(data: list) -> Dataset:
    samples = []
    for item in data:
        ents = item.get("entities", [])
        ents_fmt = [
            {"text": e["text"], "label": e["label"]}
            for e in ents if e["label"] in LABELS
        ]
        prompt = (
            f"{INSTRUCTION}\n"
            f"Texto: {item['text']}\n"
            f"Entidades: {json.dumps(ents_fmt, ensure_ascii=False)}"
        )
        samples.append({"text": prompt})

    print(f"\n[Treino] {len(samples)} prompts montados.")
    print("[Treino] Amostra de 2 prompts — 1 com entidade, 1 sem:")
    com_ent = next((s for s in samples if "label" in s["text"]), None)
    sem_ent = next((s for s in samples if '"entities": []' in s["text"]
                    or s["text"].endswith("[]")), None)
    for label, sample in [("COM entidade", com_ent), ("SEM entidade", sem_ent)]:
        if sample:
            print(f"\n  [{label}]")
            for linha in sample["text"].splitlines():
                print(f"    {linha[:100]}")

    return Dataset.from_list(samples)


def prepare_test_dataset(data: list) -> Dataset:
    samples = []
    for item in data:
        ents = item.get("entities", [])
        ents_fmt = [
            {"text": e["text"], "label": e["label"], "token_indices": e.get("token_indices", [])}
            for e in ents if e["label"] in LABELS
        ]
        samples.append({
            "input_text":    item["text"],
            "tokens":        json.dumps(item.get("tokens", []), ensure_ascii=False),
            "true_entities": json.dumps(ents_fmt, ensure_ascii=False),
        })

    print(f"\n[Teste] {len(samples)} sentenças no dataset de teste.")
    print("[Teste] Amostra das primeiras 3 entradas:")
    for i, s in enumerate(samples[:3]):
        ents = json.loads(s["true_entities"])
        print(f"  Entrada {i+1}:")
        print(f"    input_text    : {s['input_text'][:80]!r}"
              f"{'...' if len(s['input_text']) > 80 else ''}")
        print(f"    true_entities : {ents}")
    sem_ent = sum(1 for s in samples if s["true_entities"] == "[]")
    print(f"[Teste] Sentenças sem entidade: {sem_ent}/{len(samples)} "
          f"({100*sem_ent/len(samples):.1f}%)\n")

    return Dataset.from_list(samples)


def parse_model_output(output_text: str) -> list:
    output_text = re.sub(r"<think>.*?</think>",     "", output_text, flags=re.DOTALL)
    output_text = re.sub(r"<thought>.*?</thought>", "", output_text, flags=re.DOTALL)
    output_text = re.sub(r"```json|```",            "", output_text)
    output_text = output_text.strip()

    start = output_text.find("[")
    if start == -1:
        if output_text:
            print(f"[Parse] Nenhum array JSON encontrado. Saída bruta: {output_text[:120]!r}")
        return []

    for end in range(start + 1, len(output_text) + 1):
        if output_text[end - 1] != "]":
            continue
        try:
            parsed = json.loads(output_text[start:end])
            break
        except json.JSONDecodeError:
            continue
    else:
        print(f"[Parse] JSONDecodeError | Saída bruta: {output_text[:120]!r}")
        return []

    result = []
    for e in parsed:
        if not isinstance(e, dict):
            print(f"[Parse] Elemento ignorado (não é dict): {e!r}")
            continue
        if "text" not in e:
            print(f"[Parse] Elemento sem campo 'text' ignorado: {e!r}")
            continue
        if e.get("label") not in LABELS:
            print(f"[Parse] Rótulo inválido ignorado: {e.get('label')!r} em {e!r}")
            continue
        result.append({"text": e["text"], "label": e["label"]})

    return result


# ============================================================
# CHECKPOINT
# ============================================================

def read_last_fold() -> int:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            try:
                return int(f.read().strip())
            except ValueError:
                return 0
    return 0


def write_last_fold(loop_idx: int):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(loop_idx))


# ============================================================
# CARREGAMENTO DO MODELO
# ============================================================

def load_model_and_tokenizer():
    lora_kwargs = dict(
        r                          = LORA_R,
        lora_alpha                 = LORA_ALPHA,
        lora_dropout               = LORA_DROPOUT,
        target_modules             = LORA_TARGETS,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
        random_state               = 42,
    )

    if CFG["api"] == "fast_language":
        from unsloth import FastLanguageModel
        from transformers import AutoTokenizer

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name    = MODEL_NAME,
            max_seq_length= MAX_LEN,
            dtype         = None,
            load_in_4bit  = True,
        )
        if not hasattr(tokenizer, "convert_tokens_to_ids"):
            tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = FastLanguageModel.get_peft_model(model, **lora_kwargs)
        model._unsloth_api = "fast_language"

    else:
        from unsloth import FastModel

        model, tokenizer = FastModel.from_pretrained(
            model_name         = MODEL_NAME,
            max_seq_length     = MAX_LEN,
            dtype              = None,
            quantization_config= NF4_CONFIG,
        )
        model = FastModel.get_peft_model(model, **lora_kwargs)
        model._unsloth_api = "fast_model"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer


def set_inference_mode(model):
    if getattr(model, "_unsloth_api", "fast_language") == "fast_model":
        from unsloth import FastModel
        FastModel.for_inference(model)
    else:
        from unsloth import FastLanguageModel
        FastLanguageModel.for_inference(model)


# ============================================================
# AVALIAÇÃO
# ============================================================

def evaluate_fold(fold_idx: int, model, tokenizer, ds_test: Dataset) -> tuple:
    """
    Métricas idênticas ao compute_metrics do BERTimbau — todas via seqeval:

      precision / recall / f1   micro  (padrão seqeval)
      precision / recall / f1   macro
      precision / recall / f1   por entidade (LOCAL, ORGANIZACAO, PESSOA, TEMPO)
      accuracy                  via seqeval.accuracy_score — mesma função do BERT

    O seqeval avalia no nível de span IOB2 completo:
    um B-X sem o I-X seguinte é penalizado como span errado.
    """
    set_inference_mode(model)

    seqeval_true: list = []
    seqeval_pred: list = []
    errors:       list = []

    for item in tqdm(ds_test, desc=f"Avaliando Fold {fold_idx}"):
        input_text    = str(item["input_text"])
        true_entities = json.loads(str(item["true_entities"]))

        prompt = f"{INSTRUCTION}\nTexto: {input_text}\nEntidades:"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        ).to(next(model.parameters()).device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW,
                use_cache=True,
            )

        new_ids     = output_ids[0][inputs["input_ids"].shape[1]:].cpu()
        output_text = tokenizer.decode(new_ids, skip_special_tokens=True)

        del inputs, output_ids, new_ids
        torch.cuda.empty_cache()

        pred_entities = parse_model_output(output_text)

        # ---- Reconstrói sequências IOB2 ----
        tokens = json.loads(str(item["tokens"]))

        # Tags verdadeiras: usa token_indices exatos salvos no load
        true_tags = ["O"] * len(tokens)
        for e in true_entities:
            idxs  = e.get("token_indices", [])
            label = e["label"]
            if not idxs:
                continue
            true_tags[idxs[0]] = f"B-{label}"
            for i in idxs[1:]:
                true_tags[i] = f"I-{label}"
        seqeval_true.append(true_tags)

        # Tags preditas: busca por sequência (modelo não retorna índices)
        # ---- normalização para busca robusta ----

        # Tags preditas: busca por sequência com normalização
        pred_tags    = ["O"] * len(tokens)
        norm_tokens  = [normalize(t) for t in tokens]
        sorted_preds = sorted(pred_entities, key=lambda e: len(e["text"].split()), reverse=True)
        for e in sorted_preds:
            ent_toks  = [normalize(t) for t in e["text"].split()]
            ent_toks = [t for t in ent_toks if t]   
            if not ent_toks:                          
                continue                              
            label     = e["label"]
            n         = len(ent_toks)
            for i in range(len(norm_tokens) - n + 1):
                if norm_tokens[i:i+n] == ent_toks and all(pred_tags[i+k] == "O" for k in range(n)):
                    pred_tags[i] = f"B-{label}"
                    for k in range(1, n):
                        pred_tags[i+k] = f"I-{label}"
                    break
        seqeval_pred.append(pred_tags)

        # ---- Registro de erros ----
        true_list = [(e["text"], e["label"]) for e in true_entities]
        pred_list = [(e["text"], e["label"]) for e in pred_entities]
        if sorted(true_list) != sorted(pred_list):
            rem_true = list(true_list)
            rem_pred = list(pred_list)
            for p in list(pred_list):
                if p in rem_true:
                    rem_true.remove(p)
                    rem_pred.remove(p)
            errors.append({
                "text":          input_text,
                "true_entities": true_entities,
                "pred_entities": pred_entities,
                "missed":        rem_true,
                "spurious":      rem_pred,
            })

    # ---- Métricas — idênticas ao compute_metrics do BERT ----
    seq_report = seq_classification_report(
        seqeval_true, seqeval_pred, output_dict=True, zero_division=0
    )

    # Por entidade — mesmo loop do BERT
    per_entity = {}
    for entity, vals in seq_report.items():
        if isinstance(vals, dict) and entity not in ("micro avg", "macro avg", "weighted avg"):
            per_entity[entity.lower()] = {
                "precision": vals["precision"],
                "recall":    vals["recall"],
                "f1":        vals["f1-score"],
            }

    metrics = {
        "seqeval": {
            # micro (padrão seqeval)
            "precision": precision_score(seqeval_true, seqeval_pred, zero_division=0),
            "recall":    recall_score(seqeval_true,    seqeval_pred, zero_division=0),
            "f1":        f1_score(seqeval_true,        seqeval_pred, zero_division=0),
            # macro
            "precision_macro": precision_score(seqeval_true, seqeval_pred, average="macro", zero_division=0),
            "recall_macro":    recall_score(seqeval_true,    seqeval_pred, average="macro", zero_division=0),
            "f1_macro":        f1_score(seqeval_true,        seqeval_pred, average="macro", zero_division=0),
            # accuracy — seqeval.accuracy_score, idêntica ao BERT
            "accuracy":  accuracy_score(seqeval_true, seqeval_pred),
            # relatório completo por entidade + avgs
            "per_label": seq_report,
            "per_entity": per_entity,
        },
    }
    return metrics, errors


# ============================================================
# MAIN
# ============================================================

def main():
    # HAREM  + cachacaNER      : col_sep="\t", file_ext=".txt"
    # leNER        : col_sep=" ",  file_ext=".conll"
    # UlyssesNER-BR: col_sep=" ",  file_ext=".conll"
    # CoNLL-2003   : col_sep=" ",  file_ext=".conll" (4 colunas; tag_col=-1 pega a última)
    DATASET_COL_SEP   = "\t"
    DATASET_FILE_EXT  = ".txt"
    DATASET_TOKEN_COL = 0
    DATASET_TAG_COL   = -1

    def _load(path: str) -> list:
        return load_data(
            path,
            col_sep   = DATASET_COL_SEP,
            token_col = DATASET_TOKEN_COL,
            tag_col   = DATASET_TAG_COL,
        )

    division_files = {i: f"particao_{i}{DATASET_FILE_EXT}" for i in ALL_FOLDS}
    all_reports    = []
    start_loop_idx = read_last_fold()

    for loop_idx, fold in enumerate(ALL_FOLDS):

        if loop_idx < start_loop_idx:
            result_path = f"resultados_fold_{fold}.json"
            if os.path.exists(result_path):
                print(f"[Checkpoint] Fold {fold} já processado. Carregando...")
                with open(result_path, "r", encoding="utf-8") as f:
                    all_reports.append(json.load(f))
                continue
            else:
                print(
                    f"[Aviso] Fold {fold} marcado como processado, "
                    "mas arquivo de resultados não encontrado. Reprocessando..."
                )
                start_loop_idx = loop_idx

        print(f"\n{'='*55}")
        print(f"  FOLD {fold} (loop {loop_idx}) — {MODEL_NAME}")
        print(f"{'='*55}")

        # ---- Dados ----
        test_data  = _load(os.path.join(BASE_DIR, division_files[fold]))
        train_data = []
        for j, fname in division_files.items():
            if j != fold:
                train_data.extend(_load(os.path.join(BASE_DIR, fname)))

        ds_train = prepare_train_dataset(train_data)
        ds_test  = prepare_test_dataset(test_data)

        # ---- Modelo ----
        model, tokenizer = load_model_and_tokenizer()

        # ---- Treino ----
        steps_per_epoch = max(1, len(ds_train) // (BATCH_SIZE * GRAD_ACCUM))
        total_steps     = steps_per_epoch * NUM_EPOCHS
        warmup_steps    = max(10, int(0.05 * total_steps))

        sft_cfg = SFTConfig(
            output_dir                  = f"modelos_por_fold/fold_{fold}",
            per_device_train_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = GRAD_ACCUM,
            num_train_epochs            = NUM_EPOCHS,
            learning_rate               = LR,
            warmup_steps                = warmup_steps,
            logging_steps               = 10,
            bf16                        = True,
            fp16                        = False,
            save_strategy               = "no",
            eval_strategy               = "no",
            report_to                   = "none",
            push_to_hub                 = False,
            max_seq_length              = MAX_LEN,
            dataset_text_field          = "text",
            dataloader_num_workers      = 0,
        )

        trainer = SFTTrainer(
            model         = model,
            tokenizer     = tokenizer,
            train_dataset = ds_train,
            args          = sft_cfg,
        )
        trainer.train()
        trainer.state.log_history.clear()
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

        # ---- Avaliação ----
        metrics, errors = evaluate_fold(fold, model, tokenizer, ds_test)

        # ---- Salvar erros ----
        with open(f"erros_fold_{fold}.txt", "w", encoding="utf-8") as f_err:
            for err in errors:
                f_err.write("-" * 80 + "\n")
                f_err.write(f"Texto:\n{err['text']}\n\n")
                f_err.write("Entidades verdadeiras:\n")
                f_err.write(json.dumps(err["true_entities"], ensure_ascii=False, indent=2))
                f_err.write("\n\nEntidades previstas:\n")
                f_err.write(json.dumps(err["pred_entities"], ensure_ascii=False, indent=2))
                f_err.write(f"\n\nPerdidas  (FN): {err['missed']}")
                f_err.write(f"\nEspúrias  (FP): {err['spurious']}\n\n")

        # ---- Salvar métricas ----
        result_path = f"resultados_fold_{fold}.json"
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

        # ---- Relatório por fold ----
        sv      = metrics["seqeval"]
        per_lbl = sv["per_label"]
        seq_ma  = per_lbl.get("macro avg", {})
        seq_mi  = per_lbl.get("micro avg", {})

        with open("relatorio_folds.txt", "a", encoding="utf-8") as f:
            f.write(f"\n====== FOLD {fold} — {MODEL_NAME} ======\n")
            f.write("  seqeval por entidade:\n")
            for label in LABELS:
                r = per_lbl.get(label, {})
                f.write(
                    f"    {label}: "
                    f"P={r.get('precision', 0):.3f}  "
                    f"R={r.get('recall', 0):.3f}  "
                    f"F1={r.get('f1-score', 0):.3f}\n"
                )
            f.write(
                f"\n  Macro Avg : P={seq_ma.get('precision',0):.3f}  "
                f"R={seq_ma.get('recall',0):.3f}  F1={seq_ma.get('f1-score',0):.3f}\n"
                f"  Micro Avg : P={seq_mi.get('precision',0):.3f}  "
                f"R={seq_mi.get('recall',0):.3f}  F1={seq_mi.get('f1-score',0):.3f}\n"
                f"  Accuracy  : {sv['accuracy']:.3f}\n"
                + "=" * 55 + "\n"
            )

        all_reports.append(metrics)
        write_last_fold(loop_idx + 1)

        # ---- Liberar VRAM ----
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    # ============================================================
    # RELATÓRIO FINAL
    # ============================================================
    n = len(all_reports)
    if n == 0:
        print("Nenhum fold processado.")
        return

    def avg(key_path: list) -> float:
        total = 0.0
        valid = 0
        for rep in all_reports:
            node = rep
            try:
                for k in key_path:
                    node = node[k]
                total += float(node)
                valid += 1
            except (KeyError, TypeError):
                pass
        return total / valid if valid > 0 else 0.0

    final_per_label = {}
    for label in LABELS:
        final_per_label[label] = {
            "precision": avg(["seqeval", "per_label", label, "precision"]),
            "recall":    avg(["seqeval", "per_label", label, "recall"]),
            "f1":        avg(["seqeval", "per_label", label, "f1-score"]),
        }

    report_lines = [
        f"=== MÉDIAS DOS {n} FOLDS ===",
        f"Modelo : {MODEL_NAME}",
        f"(Fold {HPARAM_FOLD} excluído — reservado para hiperparâmetros do BERT)",
        f"Hiperparâmetros: épocas={NUM_EPOCHS} | lr={LR} | batch={BATCH_SIZE} "
        f"| grad_accum={GRAD_ACCUM} | r={LORA_R} | alpha={LORA_ALPHA} | dropout={LORA_DROPOUT}",
        "",
        "  Por entidade (média dos folds):",
    ]
    for label, s in final_per_label.items():
        report_lines.append(
            f"    {label}: P={s['precision']:.3f}  R={s['recall']:.3f}  F1={s['f1']:.3f}"
        )
    report_lines += [
        "",
        f"  Macro Avg : P={avg(['seqeval','per_label','macro avg','precision']):.3f}  "
        f"R={avg(['seqeval','per_label','macro avg','recall']):.3f}  "
        f"F1={avg(['seqeval','per_label','macro avg','f1-score']):.3f}",
        f"  Micro Avg : P={avg(['seqeval','per_label','micro avg','precision']):.3f}  "
        f"R={avg(['seqeval','per_label','micro avg','recall']):.3f}  "
        f"F1={avg(['seqeval','per_label','micro avg','f1-score']):.3f}",
        f"  Accuracy  : {avg(['seqeval','accuracy']):.3f}",
    ]

    report_text = "\n".join(report_lines) + "\n"

    with open("relatorio_final.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    print(report_text)
    print(f"✅ Concluído. Modelo: {MODEL_NAME} | Folds processados: {n}")


if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/juliaribeiro/venv-ner/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!

  FOLD 1 (loop 0) — unsloth/Qwen3.5-9B
[CoNLL] I- sem B- anterior ('I-NOME_ORGANIZACAO' apos 'None') em: "de Aguardente"

[CoNLL] ../Partitions/Datasets/cachacaNER/particao_1.txt: 1339 sentenças carregadas.
[CoNLL] Amostra das primeiras 3 sentenças lidas:
  Sentença 1:
    text     : 'NOME DA CACHAÇA : Porto Estrela Ouro 1 Litro'
    entities : [{'text': 'Porto Estrela', 'label': 'NOME_BEBIDA', 'token_indices': [4, 5]}, {'text': 'Ouro', 'label': 'CLASSIFICACAO_BEBIDA', 'token_indices': [6]}, {'text': '1 Litro', 'label': 'VOLUME', 'token_indices': [7, 8]}]
  Sentença 2:
    text     : 'PREÇO : R$ 150 , 0'
    entities : [{'text': 'R$ 150 , 0', 'label': 'PRECO', 'token_indices': [2, 3, 4, 5]}]
  Sentença 3:
    text     : 'DESCRIÇÃO DA CACHAÇA : A Cachaça Porto Estrela é produzida na cidade de Pedra

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|███████████████████████| 760/760 [00:02<00:00, 367.81it/s]
[transformers] Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth: Tokenizing ["text"] (num_proc=20): 100%|█| 11195/11195 [00:35<00:00, 31
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,195 | Nu

Step,Training Loss
10,1.795887
20,1.824755
30,1.793018
40,1.635396
50,1.554699
60,1.451124
70,1.258496
80,1.002395
90,0.659471
100,0.453209


Avaliando Fold 1:  35%|████████████▎                      | 469/1339 [2:37:28<4:47:00, 19.79s/it]